In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

### Modeling Preparation

In [2]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
import numpy as np

In [3]:
train_df = pd.read_csv("../data/raw/train.csv")
train_df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
train_df = pd.read_csv("../data/processed/train_processed.csv")


In [5]:

print("Shape:", train_df.shape)
print("Missing Values:", train_df.isnull().sum().sum())

Shape: (891, 38)
Missing Values: 0


In [6]:
X = train_df.drop(["Survived", "PassengerId"], axis=1)
y = train_df["Survived"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (891, 36)
y shape: (891,)


### Train Test Split

In [7]:
from sklearn.model_selection import train_test_split


In [8]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [9]:

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)

X_train: (712, 36)
X_val: (179, 36)
y_train: (712,)
y_val: (179,)


### Feature Scaling

In [10]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()



In [11]:
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

print("X_train_scaled shape:", X_train_scaled.shape)
print("X_val_scaled shape:", X_val_scaled.shape)

X_train_scaled shape: (712, 36)
X_val_scaled shape: (179, 36)


### Final Sanity Check

In [12]:
print("NaNs in X_train_scaled:", np.isnan(X_train_scaled).sum())
print("NaNs in X_val_scaled:", np.isnan(X_val_scaled).sum())

print("Infinite values in X_train_scaled:", np.isinf(X_train_scaled).sum())
print("Infinite values in X_val_scaled:", np.isinf(X_val_scaled).sum())

NaNs in X_train_scaled: 0
NaNs in X_val_scaled: 0
Infinite values in X_train_scaled: 0
Infinite values in X_val_scaled: 0


## Train Logistic Regression Model

We will train baseline Logistic Regression model 
using scaled features.

In [13]:
from src.model import train_logistic_regression

# Train model using scaled data
log_model = train_logistic_regression(X_train_scaled, y_train)

print("Logistic Regression model trained successfully.")

Logistic Regression model trained successfully.


### Model Evaluation (Logistic Regression)

In [14]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [15]:

# Predict on validation data
y_val_pred = log_model.predict(X_val_scaled)

# Accuracy
accuracy = accuracy_score(y_val, y_val_pred)
print("Validation Accuracy:", accuracy)

# Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred))

# Classification Report
print("\nClassification Report:")
print(classification_report(y_val, y_val_pred))

Validation Accuracy: 0.8100558659217877

Confusion Matrix:
[[89 16]
 [18 56]]

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.85      0.84       105
           1       0.78      0.76      0.77        74

    accuracy                           0.81       179
   macro avg       0.80      0.80      0.80       179
weighted avg       0.81      0.81      0.81       179



In [16]:
from sklearn.metrics import accuracy_score

train_acc = accuracy_score(y_train, log_model.predict(X_train_scaled))
print("Training Accuracy:", train_acc)

Training Accuracy: 0.8567415730337079


### Random Forest Model

In [17]:
from src.model import train_random_forest

# Train model (using unscaled data)
rf_model = train_random_forest(X_train, y_train)

print("Random Forest trained successfully.")

Random Forest trained successfully.


###  Random Forest Evaluation

In [18]:
# Predict
y_val_pred_rf = rf_model.predict(X_val)

# Accuracy
rf_accuracy = accuracy_score(y_val, y_val_pred_rf)
print("Random Forest Validation Accuracy:", rf_accuracy)

# Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred_rf))

# Classification Report
print("\nClassification Report:")
print(classification_report(y_val, y_val_pred_rf))

Random Forest Validation Accuracy: 0.8156424581005587

Confusion Matrix:
[[92 13]
 [20 54]]

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.88      0.85       105
           1       0.81      0.73      0.77        74

    accuracy                           0.82       179
   macro avg       0.81      0.80      0.81       179
weighted avg       0.82      0.82      0.81       179



In [19]:
train_acc_rf = accuracy_score(y_train, rf_model.predict(X_train))
print("RF Training Accuracy:", train_acc_rf)

RF Training Accuracy: 0.851123595505618


###  Model Comparison Summary

In [20]:

comparison_df = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest (Regularized)"],
    "Training Accuracy": [train_acc, train_acc_rf],
    "Validation Accuracy": [accuracy, rf_accuracy]
})

comparison_df

,Model,Training Accuracy,Validation Accuracy
0,Logistic Regression,0.856742,0.810056
1,Random Forest (Regularized),0.851124,0.815642


### Gradient Boosting Model

In [21]:
from src.model import train_gradient_boosting

# Train model (using unscaled features)
gb_model = train_gradient_boosting(X_train, y_train)

print("Gradient Boosting trained successfully.")

Gradient Boosting trained successfully.


###  Gradient Boosting Evaluation


In [22]:

# TRAINING ACCURACY 
train_acc_gb = gb_model.score(X_train, y_train)
print("Gradient Boosting Training Accuracy:", train_acc_gb)


# VALIDATION PREDICTION
y_val_pred_gb = gb_model.predict(X_val)


# VALIDATION ACCURACY
gb_accuracy = accuracy_score(y_val, y_val_pred_gb)
print("Gradient Boosting Validation Accuracy:", gb_accuracy)


# CONFUSION MATRIX
print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred_gb))


# CLASSIFICATION REPORT
print("\nClassification Report:")
print(classification_report(y_val, y_val_pred_gb))

Gradient Boosting Training Accuracy: 0.8932584269662921
Gradient Boosting Validation Accuracy: 0.8100558659217877

Confusion Matrix:
[[90 15]
 [19 55]]

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.86      0.84       105
           1       0.79      0.74      0.76        74

    accuracy                           0.81       179
   macro avg       0.81      0.80      0.80       179
weighted avg       0.81      0.81      0.81       179



In [23]:
!pip install catboost

### CatBoost Model

In [24]:
from src.model import train_catboost

# Train CatBoost (using unscaled data)
cat_model = train_catboost(X_train, y_train)

print("CatBoost trained successfully.")

CatBoost trained successfully.


### CatBoost Evaluation


In [25]:
# TRAINING ACCURACY
train_acc_cat = cat_model.score(X_train, y_train)
print("CatBoost Training Accuracy:", train_acc_cat)


# VALIDATION PREDICTION
y_val_pred_cat = cat_model.predict(X_val)


# VALIDATION ACCURACY

cat_accuracy = accuracy_score(y_val, y_val_pred_cat)
print("CatBoost Validation Accuracy:", cat_accuracy)


# CONFUSION MATRIX
print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred_cat))


# CLASSIFICATION REPORT
print("\nClassification Report:")
print(classification_report(y_val, y_val_pred_cat))

CatBoost Training Accuracy: 0.9606741573033708
CatBoost Validation Accuracy: 0.8379888268156425

Confusion Matrix:
[[93 12]
 [17 57]]

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.89      0.87       105
           1       0.83      0.77      0.80        74

    accuracy                           0.84       179
   macro avg       0.84      0.83      0.83       179
weighted avg       0.84      0.84      0.84       179



### Baseline Model Comparison

In [26]:
from sklearn.metrics import f1_score
import pandas as pd

# F1 Scores (Class 1)
f1_log = f1_score(y_val, y_val_pred)
f1_rf = f1_score(y_val, y_val_pred_rf)
f1_gb = f1_score(y_val, y_val_pred_gb)
f1_cat = f1_score(y_val, y_val_pred_cat)


# COMPARISON TABLE
comparison_df = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "Gradient Boosting",
        "CatBoost"
    ],
    "Train Accuracy": [
        train_acc,
        train_acc_rf,
        train_acc_gb,
        train_acc_cat
    ],
    "Validation Accuracy": [
        accuracy,
        rf_accuracy,
        gb_accuracy,
        cat_accuracy
    ],
    "F1 Score (Class 1)": [
        f1_log,
        f1_rf,
        f1_gb,
        f1_cat
    ]
})

comparison_df.sort_values(by="Validation Accuracy", ascending=False)

,Model,Train Accuracy,Validation Accuracy,F1 Score (Class 1)
3,CatBoost,0.960674,0.837989,0.797203
1,Random Forest,0.851124,0.815642,0.765957
0,Logistic Regression,0.856742,0.810056,0.767123
2,Gradient Boosting,0.893258,0.810056,0.763889


In [27]:
print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("y_train shape:", y_train.shape)
print("y_val shape:", y_val.shape)

print("\nTarget distribution (train):")
print(y_train.value_counts(normalize=True))

print("\nTarget distribution (val):")
print(y_val.value_counts(normalize=True))

X_train shape: (712, 36)
X_val shape: (179, 36)
y_train shape: (712,)
y_val shape: (179,)

Target distribution (train):
Survived
0    0.623596
1    0.376404
Name: proportion, dtype: float64

Target distribution (val):
Survived
0    0.586592
1    0.413408
Name: proportion, dtype: float64


### Hyperparameter Tuning – CatBoost

In [28]:
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score
import pandas as pd

In [29]:
results = []

for depth in [4, 5, 6, 7, 8]:
    for lr in [0.01, 0.03, 0.05, 0.1]:
        
        model = CatBoostClassifier(
            depth=depth,
            learning_rate=lr,
            iterations=300,
            loss_function='Logloss',
            eval_metric='Accuracy',
            random_state=42,
            verbose=False,
            early_stopping_rounds=50
        )
        
        model.fit(X_train, y_train, eval_set=(X_val, y_val))
        
        train_acc = accuracy_score(y_train, model.predict(X_train))
        val_acc = accuracy_score(y_val, model.predict(X_val))
        
        results.append([depth, lr, train_acc, val_acc])

results_df = pd.DataFrame(
    results,
    columns=["depth", "learning_rate", "train_acc", "val_acc"]
)

results_df.sort_values("val_acc", ascending=False).head()

,depth,learning_rate,train_acc,val_acc
11,6,0.10,0.908708,0.849162
19,8,0.10,0.872191,0.843575
15,7,0.10,0.893258,0.843575
16,8,0.01,0.846910,0.837989
14,7,0.05,0.844101,0.837989


In [30]:
results_iter = []

for it in [200, 300, 400, 500, 700, 1000]:
    
    model = CatBoostClassifier(
        depth=6,
        learning_rate=0.10,
        iterations=it,
        loss_function='Logloss',
        eval_metric='Accuracy',
        random_state=42,
        verbose=False,
        early_stopping_rounds=50
    )
    
    model.fit(X_train, y_train, eval_set=(X_val, y_val))
    
    train_acc = accuracy_score(y_train, model.predict(X_train))
    val_acc = accuracy_score(y_val, model.predict(X_val))
    
    results_iter.append([it, train_acc, val_acc])

results_iter_df = pd.DataFrame(
    results_iter,
    columns=["iterations", "train_acc", "val_acc"]
)

results_iter_df.sort_values("val_acc", ascending=False)

,iterations,train_acc,val_acc
0,200,0.908708,0.849162
1,300,0.908708,0.849162
2,400,0.908708,0.849162
3,500,0.908708,0.849162
4,700,0.908708,0.849162
5,1000,0.908708,0.849162


### Tuning l2_leaf_reg (Regularization)

Locked Parameters:
- depth = 6
- learning_rate = 0.10
- iterations = 300


In [31]:
results_reg = []

for reg in [1, 3, 5, 7, 9, 15, 25]:
    
    model = CatBoostClassifier(
        depth=6,
        learning_rate=0.10,
        iterations=300,
        l2_leaf_reg=reg,
        loss_function='Logloss',
        eval_metric='Accuracy',
        random_state=42,
        verbose=False,
        early_stopping_rounds=50
    )
    
    model.fit(X_train, y_train, eval_set=(X_val, y_val))
    
    train_acc = accuracy_score(y_train, model.predict(X_train))
    val_acc = accuracy_score(y_val, model.predict(X_val))
    
    results_reg.append([reg, train_acc, val_acc])

results_reg_df = pd.DataFrame(
    results_reg,
    columns=["l2_leaf_reg", "train_acc", "val_acc"]
)

results_reg_df.sort_values("val_acc", ascending=False)

,l2_leaf_reg,train_acc,val_acc
2,5,0.873596,0.854749
0,1,0.915730,0.849162
1,3,0.908708,0.849162
3,7,0.898876,0.843575
4,9,0.834270,0.837989
5,15,0.859551,0.837989
6,25,0.862360,0.832402


### LOCKED PARAMETERS
depth = 6
learning_rate = 0.10
iterations = 300
l2_leaf_reg = 5

In [32]:
from sklearn.model_selection import cross_val_score


### Cross-Validation Stability Check (5-Fold)


In [33]:

final_model = CatBoostClassifier(
    depth=6,
    learning_rate=0.10,
    iterations=300,
    l2_leaf_reg=5,
    loss_function='Logloss',
    eval_metric='Accuracy',
    random_state=42,
    verbose=False
)

cv_scores = cross_val_score(
    final_model,
    X,
    y,
    cv=5,
    scoring='accuracy'
)

print("Fold Accuracies:", cv_scores)
print("Mean Accuracy:", np.mean(cv_scores))
print("Standard Deviation:", np.std(cv_scores))

Fold Accuracies: [0.83240223 0.81460674 0.87640449 0.78089888 0.85393258]
Mean Accuracy: 0.831648986253217
Standard Deviation: 0.03276032168630293


### ## 6.3 Feature Importance Re-Evaluation

Objective:
Analyze feature importance of the optimized CatBoost model 
and identify weak/noisy features for potential removal.

In [34]:
# Train final tuned model on full training data
final_model.fit(X, y)

# Get feature importance
importances = final_model.get_feature_importance()

feature_importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": importances
}).sort_values(by="importance", ascending=False)

feature_importance_df.head(15)

,feature,importance
1,Age,17.939199
4,Fare,17.491014
11,Title_Mr,15.759542
0,Pclass,4.730182
7,Sex_male,4.683803
5,FamilySize,3.463426
2,SibSp,3.455810
34,Class_Sex_3_female,2.830069
28,FareGroup_MidLow,2.762293
9,Embarked_S,2.677481


In [35]:
feature_importance_df.tail(10)

,feature,importance
14,FamilyCategory_Large,0.655540
26,AgeGroup_MidAge,0.605144
17,Deck_C,0.599763
6,IsAlone,0.454644
16,Deck_B,0.322629
24,AgeGroup_Teen,0.261099
27,AgeGroup_Senior,0.253523
21,Deck_G,0.113222
22,Deck_T,0.029538
20,Deck_F,0.000000


### Removing Extremely Low Importance Features

In [36]:
X_reduced = X.drop(columns=["Deck_F", "Deck_T"])

### Re-training After Feature Pruning

In [37]:
from src.model import build_catboost
from src.config import CATBOOST_PARAMS

model = build_catboost(CATBOOST_PARAMS)

In [38]:
cv_scores_reduced = cross_val_score(
    model,
    X_reduced,
    y,
    cv=5,
    scoring='accuracy'
)

print("Fold Accuracies:", cv_scores_reduced)
print("Mean Accuracy:", np.mean(cv_scores_reduced))
print("Standard Deviation:", np.std(cv_scores_reduced))

Fold Accuracies: [0.82681564 0.82022472 0.85955056 0.76404494 0.85393258]
Mean Accuracy: 0.824913690289373
Standard Deviation: 0.03397769046097492


## 6.5 Small Ensemble (CatBoost + Logistic Regression)

Objective:
Combine predictions from tuned CatBoost and Logistic Regression
to improve generalization performance.

In [39]:
from sklearn.linear_model import LogisticRegression

In [40]:

# Logistic Model
log_model = LogisticRegression(max_iter=1000)

# Ensemble via soft voting (manual averaging)
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)

ensemble_scores = []

for train_idx, val_idx in kf.split(X_reduced):
    
    X_tr, X_val_fold = X_reduced.iloc[train_idx], X_reduced.iloc[val_idx]
    y_tr, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
    
    # Train models
    cat_model = build_catboost(CATBOOST_PARAMS)
    cat_model.fit(X_tr, y_tr)
    
    log_model.fit(X_tr, y_tr)
    
    # Predict probabilities
    cat_pred = cat_model.predict_proba(X_val_fold)[:,1]
    log_pred = log_model.predict_proba(X_val_fold)[:,1]
    
    # Weighted average (start simple 75/25)
    final_pred = 0.75 * cat_pred + 0.25 * log_pred
    
    final_pred_binary = (final_pred > 0.5).astype(int)
    
    acc = np.mean(final_pred_binary == y_val_fold)
    ensemble_scores.append(acc)

print("Fold Accuracies:", ensemble_scores)
print("Mean Accuracy:", np.mean(ensemble_scores))
print("Std Deviation:", np.std(ensemble_scores))

Fold Accuracies: [np.float64(0.8268156424581006), np.float64(0.8258426966292135), np.float64(0.8707865168539326), np.float64(0.8033707865168539), np.float64(0.8314606741573034)]
Mean Accuracy: 0.8316552633230808
Std Deviation: 0.02185640573254731


In [41]:

kf = KFold(n_splits=5, shuffle=True, random_state=42)

ensemble_scores_80 = []

for train_idx, val_idx in kf.split(X_reduced):
    
    X_tr, X_val_fold = X_reduced.iloc[train_idx], X_reduced.iloc[val_idx]
    y_tr, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
    
    # Models
    cat_model = build_catboost(CATBOOST_PARAMS)
    cat_model.fit(X_tr, y_tr)
    
    log_model = LogisticRegression(max_iter=1000)
    log_model.fit(X_tr, y_tr)
    
    # Probabilities
    cat_pred = cat_model.predict_proba(X_val_fold)[:,1]
    log_pred = log_model.predict_proba(X_val_fold)[:,1]
    
    # 80 / 20 weight
    final_pred = 0.80 * cat_pred + 0.20 * log_pred
    
    final_binary = (final_pred > 0.5).astype(int)
    
    acc = np.mean(final_binary == y_val_fold)
    ensemble_scores_80.append(acc)

print("Fold Accuracies:", ensemble_scores_80)
print("Mean Accuracy:", np.mean(ensemble_scores_80))
print("Std Deviation:", np.std(ensemble_scores_80))

Fold Accuracies: [np.float64(0.8268156424581006), np.float64(0.8258426966292135), np.float64(0.8595505617977528), np.float64(0.8033707865168539), np.float64(0.8314606741573034)]
Mean Accuracy: 0.8294080723118448
Std Deviation: 0.017945226076140543


In [42]:
from src.model import build_catboost, build_logistic, ensemble_predict
from src.config import CATBOOST_PARAMS, ENSEMBLE_WEIGHTS

# Training on full reduced data
cat_model = build_catboost(CATBOOST_PARAMS)
log_model = build_logistic()

cat_model.fit(X_reduced, y)
log_model.fit(X_reduced, y)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [43]:
import os
os.makedirs("models", exist_ok=True)
print("models folder ready")

models folder ready


In [44]:
import joblib

# Saving CatBoost
cat_model.save_model("models/catboost_final.cbm")

# Saving Logistic
joblib.dump(log_model, "models/logistic_final.pkl")

print("Models saved successfully.")

Models saved successfully.


### ALIGNMENT REBUILD

In [45]:
print(train_df.shape)

(891, 38)


In [46]:
X = train_df.drop(columns=["Survived"])
y = train_df["Survived"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (891, 37)
y shape: (891,)


In [47]:
from src.config import CATBOOST_PARAMS

cat_model = build_catboost(CATBOOST_PARAMS)
log_model = build_logistic()

In [50]:
from src.config import CATBOOST_PARAMS

cat_model = build_catboost(CATBOOST_PARAMS)
log_model = build_logistic()

In [54]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
import numpy as np

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cat_scores = []

for train_idx, val_idx in skf.split(X, y):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = build_catboost(CATBOOST_PARAMS)
    model.fit(X_train, y_train)

    preds = model.predict(X_val)
    score = accuracy_score(y_val, preds)
    cat_scores.append(score)

print("CatBoost CV Mean:", np.mean(cat_scores))
print("CatBoost CV Std:", np.std(cat_scores))

CatBoost CV Mean: 0.8282656455966354
CatBoost CV Std: 0.026593037431321263


In [55]:
final_cat = build_catboost(CATBOOST_PARAMS)
final_cat.fit(X, y)

print("Final CatBoost trained.")

Final CatBoost trained.


In [56]:
final_log = build_logistic()
final_log.fit(X, y)

print("Final Logistic trained.")

Final Logistic trained.


/Users/kashakjoshi/Desktop/titanic/venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [57]:
final_cat.save_model("../models/catboost_final.cbm")

import joblib
joblib.dump(final_log, "../models/logistic_final.pkl")

print("Clean models saved.")

Clean models saved.
